# V1

In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

In [43]:


def preprocess_data(file_path: str, categorical_cols: list, label_cols: list, delete_cols: list = None):
    """
    一个更灵活的数据预处理流程：
    1. 加载数据
    2. 转换数据类型并处理缺失值
    3. 对指定的类别型变量进行独热编码
    4. 根据用户定义的列表分离出特征(X)和标签(y)
    5. 分割数据集
    6. 标准化数值型特征
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # 删除用户指定的列（如果有）
    if delete_cols:
        for col in delete_cols:
            if col in df.columns:
                df.drop(columns=[col], inplace=True)
                print(f"   - 已删除列: {col}")
            else:
                print(f"   - 警告：尝试删除的列 '{col}' 不存在于数据集中。")
        print(f"✅ 删除指定列后，数据集维度为: {df.shape}")

    # --- 步骤 2: 确保数据类型正确并处理缺失值 ---
    print("\n🔄 正在处理数据类型和缺失值...")
    
    # 确定哪些列需要被当作数值型处理
    # 即：所有列中，排除掉用户指定的“类别型”和“标签”列
    numeric_cols_initial = [
        col for col in df.columns if col not in categorical_cols and col not in label_cols
    ]

    for col in numeric_cols_initial:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
    
    # 用 'missing' 填充类别列中的缺失值
    for col in categorical_cols:
        if col in df.columns and df[col].isnull().any():
            df[col].fillna('missing', inplace=True)

    print("✅ 数据类型处理和缺失值填充完成。")

    # --- 步骤 3: 对类别型变量进行独热编码 (在分离X和y之前) ---
    print("\n🔄 正在进行独热编码...")
    
    # 确保所有指定的类别列都存在于DataFrame中
    actual_categorical_cols = [col for col in categorical_cols if col in df.columns]
    if not actual_categorical_cols:
        print("   - 未找到需要独热编码的类别列。")
        df_encoded = df
    else:
        print(f"   - 将对以下列进行独热编码: {actual_categorical_cols}")
        df_encoded = pd.get_dummies(df, columns=actual_categorical_cols, prefix=actual_categorical_cols)
        print(f"✅ 独热编码完成。数据集新维度: {df_encoded.shape}")

    # --- 步骤 4: 创建最终的标签(y)和特征(X) ---
    print("\n🔄 正在分离特征(X)和标签(y)...")
    
    # 智能地构建最终的标签列名列表
    # 如果用户指定的某个标签列被独热编码了，我们需要找到所有编码后的列
    final_label_cols = []
    for col in label_cols:
        if col in actual_categorical_cols: # 如果这个标签列刚刚被编码了
            # 找到所有以它为前缀的新列
            encoded_cols = [c for c in df_encoded.columns if c.startswith(f"{col}_")]
            final_label_cols.extend(encoded_cols)
            print(f"   - 标签列 '{col}' 已被独热编码为: {encoded_cols}")
        elif col in df_encoded.columns: # 如果这个标签列是数值型或未被编码的
            final_label_cols.append(col)
        else:
            print(f"   - 警告：指定的标签列 '{col}' 不在最终数据中，将被忽略。")
    
    if not final_label_cols:
        print("❌ 错误：未能构建有效的标签列。请检查您的LABEL_COLUMNS配置。")
        return None

    print(f"   - 最终用于标签(y)的列: {final_label_cols}")
    
    y = df_encoded[final_label_cols]
    X = df_encoded.drop(columns=final_label_cols)
    print("✅ 特征和标签分离完成。")

    # --- 步骤 5: 数据集分割 ---
    print("\n🔄 正在划分数据集...")
    try:
        # 对于多标签y，分层抽样(stratify)会变得复杂，这里暂时不使用
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        print("✅ 数据集已成功划分为训练集和测试集。")
        print(f"   - 训练集大小: {X_train.shape[0]} 行")
        print(f"   - 测试集大小: {X_test.shape[0]} 行")
    except Exception as e:
        print(f"❌ 数据集分割失败: {e}")
        return None

    # --- 步骤 6: 标准化数值型特征 ---
    print("\n🔄 正在标准化数值型特征...")
    
    # 重新确定哪些是数值列 (即所有非独热编码生成的列)
    numeric_cols_final = [
        col for col in X_train.columns 
        if X_train[col].dtype != 'uint8' # 独热编码生成的列是uint8类型
    ]
    
    scaler = StandardScaler()
    X_train[numeric_cols_final] = scaler.fit_transform(X_train[numeric_cols_final])
    X_test[numeric_cols_final] = scaler.transform(X_test[numeric_cols_final])
    
    print("✅ 标准化完成。")
    
    return X_train, X_test, y_train, y_test, scaler


In [44]:

# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    # 1. 指定您的完整数据集文件名
    DATASET_FILE = '../data/features/source/dataset_augmented_smotenc.csv'

    
    # 2. 在这个列表中，输入您认为是【类别型】且需要独热编码的列名
    CATEGORICAL_COLUMNS_TO_ENCODE = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]
    
    # 3. 指定一个或多个列作为您的【标签】
    #    - 如果是单列，写成 ['Level_2']
    #    - 如果是多列，写成 ['Level_2', 'RPM']
    #    - 如果标签本身是类别型且需要独热编码，只需在此处写下它的原始名称，
    #      代码会自动将其转换为编码后的多个标签列。例如，若Level_2是标签，
    #      最终的y可能会包含 Level_2_Normal, Level_2_OR 等多列。
    LABEL_COLUMNS = ['PE', 'DeLoc']

    # 4. 如果有需要删除的列，可以在这里指定
    DELETE_COLUMNS = ['OR_De_Type', 'SenLoc', 'HP']

    # ======================= 配置结束 ===========================

    # 执行完整的预处理流程
    processed_data = preprocess_data(
        file_path=DATASET_FILE,
        categorical_cols=CATEGORICAL_COLUMNS_TO_ENCODE,
        label_cols=LABEL_COLUMNS,
        delete_cols=DELETE_COLUMNS
    )
    
    if processed_data:
        X_train, X_test, y_train, y_test, scalar = processed_data
        print("\n\n🎉 --- 预处理流程全部完成 --- 🎉")
        print("您现在拥有的数据集可直接用于模型训练：")
        print(f"✅ X_train: 训练集特征, 维度 {X_train.shape}")
        print(f"✅ y_train: 训练集标签, 维度 {y_train.shape}")
        print(f"✅ X_test:  测试集特征, 维度 {X_test.shape}")
        print(f"✅ y_test:  测试集标签, 维度 {y_test.shape}")
        
        print("\n--- 最终训练集 (X_train) 预览 ---")
        print(X_train.head().to_string())

✅ 成功加载 'dataset_augmented_smotenc.csv'，数据集共有 1050 行, 35 列。
   - 已删除列: OR_De_Type
   - 已删除列: SenLoc
   - 已删除列: HP
✅ 删除指定列后，数据集维度为: (1050, 32)

🔄 正在处理数据类型和缺失值...
✅ 数据类型处理和缺失值填充完成。

🔄 正在进行独热编码...
   - 将对以下列进行独热编码: ['Freq', 'PE', 'DeLoc']
✅ 独热编码完成。数据集新维度: (1050, 38)

🔄 正在分离特征(X)和标签(y)...
   - 标签列 'PE' 已被独热编码为: ['PE_DE', 'PE_FE', 'PE_No']
   - 标签列 'DeLoc' 已被独热编码为: ['DeLoc_B', 'DeLoc_IN', 'DeLoc_IR', 'DeLoc_OR']
   - 最终用于标签(y)的列: ['PE_DE', 'PE_FE', 'PE_No', 'DeLoc_B', 'DeLoc_IN', 'DeLoc_IR', 'DeLoc_OR']
✅ 特征和标签分离完成。

🔄 正在划分数据集...
✅ 数据集已成功划分为训练集和测试集。
   - 训练集大小: 840 行
   - 测试集大小: 210 行

🔄 正在标准化数值型特征...
✅ 标准化完成。


🎉 --- 预处理流程全部完成 --- 🎉
您现在拥有的数据集可直接用于模型训练：
✅ X_train: 训练集特征, 维度 (840, 31)
✅ y_train: 训练集标签, 维度 (840, 7)
✅ X_test:  测试集特征, 维度 (210, 31)
✅ y_test:  测试集标签, 维度 (210, 7)

--- 最终训练集 (X_train) 预览 ---
           DeS       rpm       Len      Mean       RMS       Var      Skew      Kurt        CF        MF       P2P  FreqMean   FreqSTD  FreqSkew  FreqKurt  IR_Sideband_Energy_DE  IR_Sideband_Rat

In [45]:
# 创建一个字典来存放所有处理好的数据
processed_data_bundle = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'scaler': scalar  # 连标准化模型也一起保存！
}

# 使用pickle将整个字典保存到一个文件中
with open('../data/processed/processed_data.pkl', 'wb') as f:
    pickle.dump(processed_data_bundle, f)

print("\n\n🎉 --- 预处理流程全部完成 --- 🎉")
print("✅ 所有处理好的数据和标准化模型已保存到 '../data/processed/processed_data.pkl' 文件中。")



🎉 --- 预处理流程全部完成 --- 🎉
✅ 所有处理好的数据和标准化模型已保存到 'processed_data.pkl' 文件中。


# V2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

def final_preprocess_pipeline(file_path: str, label_cols: list, categorical_features: list, delete_cols: list = None):
    """
    一个完整且灵活的数据预处理流程，严格遵循用户定义。
    它将创建单列的“组合标签”以兼容后续的交叉验证。
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # --- 步骤 2: 删除用户指定的列 ---
    if delete_cols:
        # 筛选出数据集中实际存在的、需要删除的列
        cols_to_drop_existing = [col for col in delete_cols if col in df.columns]
        df.drop(columns=cols_to_drop_existing, inplace=True)
        print(f"✅ 已删除指定列: {cols_to_drop_existing}。剩余 {df.shape[1]} 列。")

    # --- 步骤 3: 创建单列的“组合标签” ---
    # 这个步骤是为了解决后续交叉验证的报错问题
    if not all(item in df.columns for item in label_cols):
        print(f"❌ 错误：您指定的某些标签列不在数据集中或已被删除。请检查列表: {label_cols}")
        return None
    
    # 将多个标签列合并成一个，用作后续的 y
    y = df[label_cols].astype(str).agg('_'.join, axis=1)
    # 从特征集中删除原始标签列
    X = df.drop(columns=label_cols)
    print(f"✅ 已成功将标签列 {label_cols} 合并为单列组合标签，用于分层抽样。")


    # --- 步骤 4: 数据类型处理和缺失值填充 ---
    print("\n🔄 正在处理特征集 X...")
    # 确定数值列（所有非明确指定的类别特征列）
    numeric_features = [col for col in X.columns if col not in categorical_features]
    for col in numeric_features:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        if X[col].isnull().any():
            X[col].fillna(X[col].median(), inplace=True)
            
    # 确定实际存在的类别特征列
    actual_categorical_features = [col for col in categorical_features if col in X.columns]
    for col in actual_categorical_features:
        if X[col].isnull().any():
            X[col].fillna('missing', inplace=True)

    # --- 步骤 5: 对类别型特征进行独热编码 ---
    print("\n🔄 正在对类别型特征进行独热编码...")
    X_encoded = pd.get_dummies(X, columns=actual_categorical_features, prefix=actual_categorical_features)
    print("✅ 特征集独热编码完成。")

    # --- 步骤 6: 数据集分割 ---
    print("\n🔄 正在划分数据集...")
    try:
        # 使用单列的组合标签 y 进行分层抽样
        X_train, X_test, y_train, y_test = train_test_split(
            X_encoded, y, 
            test_size=0.2, 
            random_state=42, 
            stratify=y
        )
        print("✅ 数据集已成功划分为训练集和测试集。")
    except Exception as e:
        print(f"❌ 数据集分割失败: {e}")
        return None

    # --- 步骤 7: 标准化数值型特征 ---
    print("\n🔄 正在标准化数值型特征...")
    scaler = StandardScaler()
    # 编码后，原始的数值列名仍然存在
    numeric_features_final = [col for col in numeric_features if col in X_train.columns]
    
    X_train[numeric_features_final] = scaler.fit_transform(X_train[numeric_features_final])
    X_test[numeric_features_final] = scaler.transform(X_test[numeric_features_final])
    print("✅ 标准化完成。")
    
    # --- 步骤 8: 使用pickle保存所有处理好的对象 ---
    processed_data_bundle = {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'scaler': scaler
    }
    with open('../data/processed/processed_data - 2.pkl', 'wb') as f:
        pickle.dump(processed_data_bundle, f)

    print("\n\n🎉 --- 预处理流程全部完成 --- 🎉")
    print("✅ 所有处理好的数据和标准化模型已保存到 '../data/processed/processed_data - 2.pkl' 文件中。")
    print(f"   - 保存的 y_train 维度是: {y_train.shape}") # 确认y是单列
    
    return processed_data_bundle

# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    # 1. 指定您的完整数据集文件名
    DATASET_FILE = '../data/features/target/dataset_augmented - 2.csv'

    
    # 2. 在这个列表中，输入您认为是【类别型】且需要独热编码的列名
    CATEGORICAL_COLUMNS_TO_ENCODE = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]
    
    # 3. 指定一个或多个列作为您的【标签】
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 4. 如果有需要删除的列，可以在这里指定
    DELETE_COLUMNS = ['SenLoc', 'HP', 'Freq', 'Len', 'OR_De_Type']
    # ======================= 配置结束 ===========================
    
    # **代码会自动处理配置中的逻辑冲突**
    # 例如，即使'PE'和'DeLoc'同时出现在LABEL_COLUMNS和CATEGORICAL_COLUMNS_TO_ENCODE中，
    # 代码也会优先将它们作为标签分离，而不会在特征集中对它们进行编码。
    # 同理，如果一个列同时出现在要编码和要删除的列表中，它会被优先删除。
    
    # 将用户定义的【标签列】从【类别特征】列表中排除，以确保逻辑清晰
    final_categorical_features = [
        col for col in CATEGORICAL_COLUMNS_TO_ENCODE if col not in LABEL_COLUMNS and col not in DELETE_COLUMNS
    ]

    final_data = final_preprocess_pipeline(
        file_path=DATASET_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_features=final_categorical_features,
        delete_cols=DELETE_COLUMNS
    )
    
    if final_data:
        print("\n--- 预处理产物预览 ---")
        print(f"X_train 维度: {final_data['X_train'].shape}")
        print(f"y_train 维度: {final_data['y_train'].shape}")
        print("\nX_train 前5行预览:")
        print(final_data['X_train'].head().to_string())

✅ 成功加载 'dataset_augmented - 2.csv'，数据集共有 2100 行, 30 列。
✅ 已删除指定列: []。剩余 30 列。
✅ 已成功将标签列 ['PE', 'DeLoc', 'DeS'] 合并为单列组合标签，用于分层抽样。

🔄 正在处理特征集 X...

🔄 正在对类别型特征进行独热编码...
✅ 特征集独热编码完成。

🔄 正在划分数据集...
✅ 数据集已成功划分为训练集和测试集。

🔄 正在标准化数值型特征...
✅ 标准化完成。


🎉 --- 预处理流程全部完成 --- 🎉
✅ 所有处理好的数据和标准化模型已保存到 'processed_data.pkl - 2' 文件中。
   - 保存的 y_train 维度是: (1680,)

--- 预处理产物预览 ---
X_train 维度: (1680, 27)
y_train 维度: (1680,)

X_train 前5行预览:
           rpm      Mean       RMS       Var      Skew      Kurt        CF        MF       P2P  FreqMean   FreqSTD  FreqSkew  FreqKurt  IR_Sideband_Energy_DE  IR_Sideband_Ratio_DE  B_Sideband_Energy_DE  B_Sideband_Ratio_DE  IR_Sideband_Energy_FE  IR_Sideband_Ratio_FE  B_Sideband_Energy_FE  B_Sideband_Ratio_FE  Env_Peak_BPFO_DE  Env_Peak_BPFI_DE  Env_Peak_BSF_DE  Env_Peak_BPFO_FE  Env_Peak_BPFI_FE  Env_Peak_BSF_FE
1873 -1.168764  0.343750 -0.681745 -0.549688 -0.072061 -0.824080 -0.740292 -0.852464 -0.760019 -0.301221 -0.959102 -0.407668  0.234677              -0.481892       

我认为对目标域的处理应该是一样的

In [3]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

def preprocess_target_domain(target_file_path: str, source_preprocessor_path: str, 
                             label_cols: list, categorical_features: list, delete_cols: list):
    """
    对目标域数据进行预处理，严格复用源域学到的规则和用户配置。
    """
    # --- 步骤 1: 加载源域的处理工具和元数据 ---
    try:
        with open(source_preprocessor_path, 'rb') as f:
            source_bundle = pickle.load(f)
        
        scaler = source_bundle['scaler']
        source_train_columns = source_bundle['X_train'].columns
        
        # 从已保存的X_train中推断出数值列的准确列表
        source_numeric_cols = [
            col for col in source_bundle['X_train'].columns 
            if source_bundle['X_train'][col].dtype != 'uint8'
        ]
        print(f"✅ 成功加载源域预处理工具 '{source_preprocessor_path}'。")

    except FileNotFoundError:
        print(f"❌ 错误：找不到源域预处理文件 '{source_preprocessor_path}'。请先运行源域的预处理脚本。")
        return None

    # --- 步骤 2: 加载目标域数据集 ---
    try:
        df_target = pd.read_csv(target_file_path)
        print(f"✅ 成功加载目标域数据 '{target_file_path}'。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{target_file_path}' 未找到。")
        return None

    # --- 步骤 3: 应用与源域完全相同的预处理步骤 ---
    
    # 3.1 删除用户指定的列
    # errors='ignore'确保即使列不存在也不会报错
    df_target.drop(columns=delete_cols, inplace=True, errors='ignore')
    
    # 3.2 从目标域数据中分离出特征（它没有真实标签）
    # 我们需要删除在源域中被当作标签的列，以保持特征集一致
    target_feature_cols = [col for col in df_target.columns if col not in label_cols]
    X_target = df_target[target_feature_cols]

    # 3.3 数据类型和缺失值处理
    print("\n🔄 正在处理数据类型和缺失值...")
    # 筛选出实际存在的类别和数值特征列
    actual_categorical_features = [col for col in categorical_features if col in X_target.columns]
    actual_numeric_features = [col for col in X_target.columns if col not in actual_categorical_features]

    for col in actual_numeric_features:
        X_target[col] = pd.to_numeric(X_target[col], errors='coerce')
        X_target[col].fillna(X_target[col].median(), inplace=True)
            
    for col in actual_categorical_features:
        X_target[col].fillna('missing', inplace=True)

    # 3.4 独热编码
    print("🔄 正在进行独热编码...")
    X_target_encoded = pd.get_dummies(X_target, columns=actual_categorical_features, prefix=actual_categorical_features)
    
    # 3.5 **关键步骤**: 列对齐
    # 使用源域训练集的列作为模板，确保目标域的列完全一致
    # fill_value=0 确保了如果目标域缺少某个类别，对应的列会被创建并填充为0
    X_target_aligned = X_target_encoded.reindex(columns=source_train_columns, fill_value=0)
    print("✅ 特征列已与源域训练集对齐。")
    
    # 3.6 **最关键步骤**: 标准化
    print("🔄 正在使用【源域的】scaler进行标准化...")
    # 只对数值列进行标准化
    # 确保只对目标域中实际存在的数值列进行操作
    numeric_cols_to_scale = [col for col in source_numeric_cols if col in X_target_aligned.columns]
    X_target_aligned[numeric_cols_to_scale] = scaler.transform(X_target_aligned[numeric_cols_to_scale])
    print("✅ 标准化完成。")
    
    # --- 步骤 4: 保存处理好的目标域特征 ---
    output_file = '../data/processed/processed_data - target.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump({'X_target': X_target_aligned}, f)
        
    print(f"\n\n🎉 --- 目标域预处理完成 --- 🎉")
    print(f"✅ 处理好的目标域特征已保存到 '{output_file}'。")
    
    return X_target_aligned

# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 (您的定义) =========================
    # 1. 指定您的目标域数据集文件名
    TARGET_DATASET_FILE = '../data/features/target/dataset_complete - 2.csv'
    
    # 2. 指定之前处理源域数据时生成的.pkl文件名
    SOURCE_PREPROCESSOR_FILE = '../data/processed/processed_data - 2.pkl'

    # 3. (从源域脚本复制) 您定义的【模型预测目标 (y)】列名
    #    这些列将从目标域特征集中被移除
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 4. (从源域脚本复制) 您定义的需要进行独热编码的【类别型特征 (X)】
    CATEGORICAL_FEATURES = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]
    
    # 5. (从源域脚本复制) 您定义的希望【彻底删除】的列名
    DELETE_COLUMNS = ['SenLoc', 'HP', 'Freq', 'Len', 'OR_De_Type']
    # ======================= 配置结束 ===========================

    # 代码会自动处理配置中的逻辑冲突
    # 例如，'PE'和'DeLoc'在CATEGORICAL_FEATURES中，但因为它们也是LABEL_COLUMNS，
    # 它们会先从特征集中被移除，因此不会参与后续的独热编码
    final_categorical_features_for_X = [
        col for col in CATEGORICAL_FEATURES if col not in LABEL_COLUMNS and col not in DELETE_COLUMNS
    ]

    processed_target_data = preprocess_target_domain(
        target_file_path=TARGET_DATASET_FILE,
        source_preprocessor_path=SOURCE_PREPROCESSOR_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_features=final_categorical_features_for_X,
        delete_cols=DELETE_COLUMNS
    )
    
    if processed_target_data is not None:
        print("\n--- 预处理产物预览 ---")
        print(f"X_target 维度: {processed_target_data.shape}")
        print("\nX_target (处理后) 前5行预览:")
        print(processed_target_data.head().to_string())

✅ 成功加载源域预处理工具 'processed_data - 2.pkl'。
✅ 成功加载目标域数据 'dataset_complete - 2.csv'。

🔄 正在处理数据类型和缺失值...
🔄 正在进行独热编码...
✅ 特征列已与源域训练集对齐。
🔄 正在使用【源域的】scaler进行标准化...
✅ 标准化完成。


🎉 --- 目标域预处理完成 --- 🎉
✅ 处理好的目标域特征已保存到 'processed_data - target.pkl'。

--- 预处理产物预览 ---
X_target 维度: (16, 27)

X_target (处理后) 前5行预览:
             rpm      Mean       RMS       Var      Skew      Kurt        CF        MF       P2P  FreqMean   FreqSTD  FreqSkew  FreqKurt  IR_Sideband_Energy_DE  IR_Sideband_Ratio_DE  B_Sideband_Energy_DE  B_Sideband_Ratio_DE  IR_Sideband_Energy_FE  IR_Sideband_Ratio_FE  B_Sideband_Energy_FE  B_Sideband_Ratio_FE  Env_Peak_BPFO_DE  Env_Peak_BPFI_DE  Env_Peak_BSF_DE  Env_Peak_BPFO_FE  Env_Peak_BPFI_FE  Env_Peak_BSF_FE
0  723744.353448 -0.619674 -0.089373 -0.198456  2.545671 -0.240246  1.208245  0.371364 -0.153930  1.653144  4.341767  1.590944  0.233069              -0.322865              3.952877              0.725461             2.293215              -0.017758              1.726459             -0.

C:\Users\17322\AppData\Local\Temp\ipykernel_39232\3227504346.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_target[col] = pd.to_numeric(X_target[col], errors='coerce')
C:\Users\17322\AppData\Local\Temp\ipykernel_39232\3227504346.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_target[col].fillna(X_target[col].median(), inplace=True)
